
# Bengali NLP Safety — V18 MuRIL OOF Multitask

**Protected Kaggle champion: V14 = 0.586**

V18 changes model family rather than doing another small threshold tweak.

### Why MuRIL?

`google/muril-base-cased` was pretrained on Indian languages including Bengali and also on
**transliterated** text. That makes it a useful diversity model for this dataset, which contains both
Bengali-script and Romanized/non-Bengali-script examples.

### What V18 does

- trains **MuRIL on all 7 labels** with grouped 5-fold OOF;
- gives `is_toxic` extra training weight because it contributes 50% of the competition score;
- reconstructs the exact probability family behind the proven **0.586 V14** submission;
- searches conservative probability/logit blends label-by-label;
- tests **script-aware `is_toxic` blending** so MuRIL may receive a different weight on
  Bengali-script vs Romanized text;
- uses completed V17 BanglaBERT/XLM-R OOF predictions only as a small optional toxic diversity component;
- does **not** replace a V14 label unless it passes a fold-stability gate.

## Run instructions

1. Open this notebook in **Google Colab**.
2. Set **Runtime → Change runtime type → T4 GPU**.
3. Run cell 1 and upload the supplied **`v18_data.zip`**.
4. Run the rest of the notebook.
5. At the end download **`v18_results.zip`**.
6. Send `v18_results.zip` to ChatGPT **before submitting any V18 CSV to Kaggle**.


In [ ]:

# 1. Upload V18 data bundle
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()
assert "v18_data.zip" in uploaded, "Please upload the supplied v18_data.zip file."

DATA_DIR = pathlib.Path("/content/v18_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v18_data.zip", "r") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ", p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install packages and preflight MuRIL
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn",
    "pandas",
    "numpy",
    "tqdm",
    "safetensors",
])

from transformers import AutoTokenizer, AutoConfig

MODEL_NAME = "google/muril-base-cased"
print("Testing access to:", MODEL_NAME)
_ = AutoConfig.from_pretrained(MODEL_NAME)
_ = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ MuRIL preflight passed.")


In [ ]:

# 3. Settings, imports, and data
import gc, re, math, random, json, zipfile, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

SEED = 42
N_FOLDS = 5
MODEL_NAME = "google/muril-base-cased"
MAX_LENGTH = 128
EPOCHS = 4
SNAPSHOT_EPOCHS = [3, 4]
TRAIN_BATCH_SIZE = 8
ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 32
LR = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

LABELS = [
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]
TARGET_WEIGHTS = np.array([3.0,1.0,1.0,1.0,1.0,1.0,1.0], dtype=np.float32)
TARGET_WEIGHTS *= len(LABELS) / TARGET_WEIGHTS.sum()

TOXIC_MIN_GAIN = 0.003
ATTRIBUTE_MIN_GAIN = 0.008
MIN_WINS = 3
MAX_LOSSES = 1

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable a T4 GPU in Colab."

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2_valid = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2_test = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
valid_folds = pd.read_csv(DATA_DIR/"validation_folds.csv", dtype={"ID":str})
champion = pd.read_csv(DATA_DIR/"submission_v14_champion.csv", dtype={"ID":str})

with np.load(DATA_DIR/"old_banglabert_predictions.npz") as z:
    old_bang_valid=z["valid"].copy()
    old_bang_test=z["test"].copy()
with np.load(DATA_DIR/"old_xlmr_predictions.npz") as z:
    old_xl_valid=z["valid"].copy()
    old_xl_test=z["test"].copy()
with np.load(DATA_DIR/"v17_banglabert_oof_test.npz") as z:
    v17_bang_oof=z["oof"].copy()
    v17_bang_test=z["test"].copy()
with np.load(DATA_DIR/"v17_xlmr_oof_test.npz") as z:
    v17_xl_oof=z["oof"].copy()
    v17_xl_test=z["test"].copy()

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert champion["ID"].tolist()==test["ID"].tolist()
assert v2_valid["ID"].tolist()==valid["ID"].tolist()
assert v2_test["ID"].tolist()==test["ID"].tolist()

def clean_text(x):
    x = unicodedata.normalize("NFC", str(x))
    return re.sub(r"\s+", " ", x).strip()

combined = pd.concat([train, valid], ignore_index=True)
combined["clean_text"] = combined["text"].map(clean_text)
test["clean_text"] = test["text"].map(clean_text)

def has_bengali(x):
    return bool(re.search(r"[\u0980-\u09FF]", str(x)))

valid_has_bn = valid["text"].map(has_bengali).to_numpy(bool)
test_has_bn = test["text"].map(has_bengali).to_numpy(bool)

print("train/valid/combined/test:", len(train), len(valid), len(combined), len(test))
print("Validation no-Bengali-script rows:", int((~valid_has_bn).sum()))
print("Test no-Bengali-script rows:", int((~test_has_bn).sum()))
display(combined[LABELS].sum().to_frame("positives"))


In [ ]:

# 4. Guaranteed balanced grouped 5-fold assignment
def normalize_group_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    return re.sub(r"\s+", " ", x).strip().lower()

combined["group_text"] = combined["text"].map(normalize_group_text)

group_stats = combined.groupby("group_text", sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos = np.array([combined[l].sum() for l in LABELS], dtype=float)
total_rows = float(len(combined))
rarity = 1.0 / np.sqrt(np.maximum(global_pos,1.0))
mat = group_stats[[f"sum_{l}" for l in LABELS]].to_numpy(float)

priority = (mat*rarity).sum(axis=1) + 0.005*group_stats["size"].to_numpy()
rng = np.random.default_rng(SEED)
order = np.argsort(-(priority + rng.random(len(group_stats))*1e-6))

fold_pos = np.zeros((N_FOLDS,len(LABELS)), dtype=float)
fold_size = np.zeros(N_FOLDS, dtype=float)
assignment = {}

for idx in order:
    row = group_stats.iloc[idx]
    vec = np.array([row[f"sum_{l}"] for l in LABELS], dtype=float)
    size = float(row["size"])
    costs = []
    for f in range(N_FOLDS):
        p = fold_pos.copy()
        s = fold_size.copy()
        p[f] += vec
        s[f] += size
        label_share = p / np.maximum(global_pos[None,:],1.0)
        size_share = s / total_rows
        label_cost = np.mean((label_share - 1/N_FOLDS)**2)
        size_cost = np.mean((size_share - 1/N_FOLDS)**2)
        costs.append(label_cost + 0.35*size_cost)
    best = int(np.argmin(costs))
    assignment[row["group_text"]] = best
    fold_pos[best] += vec
    fold_size[best] += size

combined["fold"] = combined["group_text"].map(assignment).astype(int)

counts = combined.groupby("fold").size().reindex(range(N_FOLDS), fill_value=0)
pos_table = combined.groupby("fold")[LABELS].sum().reindex(range(N_FOLDS), fill_value=0)

assert combined.groupby("group_text")["fold"].nunique().max()==1
assert (counts>0).all()
assert counts.min() >= 0.80*(len(combined)/N_FOLDS)
assert counts.max() <= 1.20*(len(combined)/N_FOLDS)
assert (pos_table.sum(axis=0).to_numpy()==combined[LABELS].sum().to_numpy()).all()

print("✅ Fold checks passed.")
print("Fold sizes:", counts.to_dict())
display(pos_table)


In [ ]:

# 5. Dataset / inference helpers
class MultiDataset(Dataset):
    def __init__(self, frame, tokenizer, with_labels=True):
        self.texts = frame["clean_text"].tolist()
        self.targets = frame[LABELS].to_numpy(np.float32) if with_labels else None
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = self.tokenizer(
            self.texts[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.targets is not None:
            enc["labels"] = self.targets[i]
        return enc

class Collator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, items):
        labels = None
        if "labels" in items[0]:
            labels = torch.tensor(np.stack([x.pop("labels") for x in items]), dtype=torch.float32)
        batch = self.tokenizer.pad(items,padding=True,pad_to_multiple_of=8,return_tensors="pt")
        if labels is not None:
            batch["labels"] = labels
        return batch

@torch.no_grad()
def predict(model, loader):
    assert len(loader.dataset)>0
    model.eval()
    parts=[]
    for batch in loader:
        batch.pop("labels", None)
        inputs={k:v.to("cuda",non_blocking=True) for k,v in batch.items()}
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inputs).logits
        parts.append(torch.sigmoid(logits.float()).cpu().numpy())
    assert parts
    out=np.concatenate(parts)
    assert np.isfinite(out).all()
    return out


In [ ]:

# 6. Train 5-fold MuRIL OOF multitask model
RUN_DIR = Path("/content/v18_output")
RUN_DIR.mkdir(exist_ok=True)

oof = np.zeros((len(combined),len(LABELS)), dtype=np.float32)
test_fold_probs=[]
histories=[]
fold_seeds=[42,3407,2026,17,1337]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

probe = tokenizer(
    combined["clean_text"].sample(min(1000,len(combined)),random_state=SEED).tolist(),
    truncation=False
)
lengths=np.array([len(x) for x in probe["input_ids"]])
print("MuRIL token p50/p95/p99:", np.percentile(lengths,[50,95,99]))
print("Fraction > MAX_LENGTH:", float((lengths>MAX_LENGTH).mean()))
del probe, lengths
gc.collect()

for fold in range(N_FOLDS):
    tr=combined[combined["fold"]!=fold].copy()
    va=combined[combined["fold"]==fold].copy()
    assert len(tr)>0 and len(va)>0
    assert set(tr["group_text"]).isdisjoint(set(va["group_text"]))

    cache=RUN_DIR/f"muril_fold_{fold}.npz"
    hist_path=RUN_DIR/f"muril_fold_{fold}_history.csv"

    if cache.exists():
        with np.load(cache,allow_pickle=False) as z:
            idx=z["va_index"]; vp=z["valid"]; tp=z["test"]
        if np.array_equal(idx,va.index.to_numpy()) and vp.shape==(len(va),len(LABELS)) and tp.shape==(len(test),len(LABELS)):
            print("✅ Reusing MuRIL fold",fold)
            oof[idx]=vp
            test_fold_probs.append(tp)
            if hist_path.exists():
                histories.extend(pd.read_csv(hist_path).to_dict("records"))
            continue

    print("\n"+"="*70)
    print(f"MuRIL fold {fold}: train={len(tr)} valid={len(va)}")
    seed_everything(fold_seeds[fold])

    collator=Collator(tokenizer)
    tr_loader=DataLoader(MultiDataset(tr,tokenizer,True),batch_size=TRAIN_BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True,collate_fn=collator)
    va_loader=DataLoader(MultiDataset(va,tokenizer,True),batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=collator)
    te_loader=DataLoader(MultiDataset(test,tokenizer,False),batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=collator)

    model=AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,num_labels=len(LABELS),problem_type="multi_label_classification"
    ).to("cuda")

    pos=tr[LABELS].sum().to_numpy(float)
    neg=len(tr)-pos
    pos_weight=np.sqrt(neg/np.maximum(pos,1))
    pos_weight=np.clip(pos_weight,0.55,4.0)
    pos_weight_t=torch.tensor(pos_weight,dtype=torch.float32,device="cuda")
    target_weight_t=torch.tensor(TARGET_WEIGHTS,dtype=torch.float32,device="cuda")

    print("pos_weight:",dict(zip(LABELS,np.round(pos_weight,3))))
    loss_fn=torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_t,reduction="none")

    no_decay=["bias","LayerNorm.weight","layer_norm.weight"]
    groups=[
        {"params":[p for n,p in model.named_parameters() if p.requires_grad and not any(x in n for x in no_decay)],"weight_decay":WEIGHT_DECAY},
        {"params":[p for n,p in model.named_parameters() if p.requires_grad and any(x in n for x in no_decay)],"weight_decay":0.0},
    ]
    optimizer=torch.optim.AdamW(groups,lr=LR)
    steps=math.ceil(len(tr_loader)/ACCUM_STEPS)*EPOCHS
    scheduler=get_linear_schedule_with_warmup(optimizer,int(steps*WARMUP_RATIO),steps)
    scaler=torch.amp.GradScaler("cuda")
    vs=[]; ts=[]; fold_hist=[]

    for epoch in range(1,EPOCHS+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running=0.0; seen=0

        for step,batch in enumerate(tqdm(tr_loader,desc=f"MuRIL fold {fold} epoch {epoch}/{EPOCHS}")):
            y=batch.pop("labels").to("cuda",non_blocking=True)
            inputs={k:v.to("cuda",non_blocking=True) for k,v in batch.items()}

            with torch.autocast(device_type="cuda",dtype=torch.float16):
                logits=model(**inputs).logits

            raw=loss_fn(logits.float(),y)
            loss=(raw*target_weight_t.unsqueeze(0)).mean()

            if not torch.isfinite(loss):
                raise FloatingPointError("Non-finite MuRIL training loss")

            scaler.scale(loss/ACCUM_STEPS).backward()
            running += float(loss.item())*len(y)
            seen += len(y)

            if (step+1)%ACCUM_STEPS==0 or step+1==len(tr_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                old_scale=scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale()>=old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        rec={"fold":fold,"epoch":epoch,"loss":running/max(seen,1)}
        print(rec)

        if epoch in SNAPSHOT_EPOCHS:
            vs.append(predict(model,va_loader))
            ts.append(predict(model,te_loader))

        fold_hist.append(rec)
        pd.DataFrame(fold_hist).to_csv(hist_path,index=False)

    vp=np.mean(vs,axis=0)
    tp=np.mean(ts,axis=0)
    assert vp.shape==(len(va),len(LABELS))
    assert tp.shape==(len(test),len(LABELS))

    oof[va.index.to_numpy()]=vp
    test_fold_probs.append(tp)
    histories.extend(fold_hist)
    np.savez_compressed(cache,va_index=va.index.to_numpy(),valid=vp,test=tp)

    del model,tr_loader,va_loader,te_loader,optimizer,scheduler,scaler
    gc.collect()
    torch.cuda.empty_cache()

assert len(test_fold_probs)==N_FOLDS
muril_test=np.mean(test_fold_probs,axis=0)
np.savez_compressed(RUN_DIR/"muril_oof_test.npz",oof=oof,test=muril_test)
pd.DataFrame(histories).to_csv(RUN_DIR/"muril_training_history.csv",index=False)
print("\n✅ MuRIL 5-fold OOF training completed.")


In [ ]:

# 7. Reconstruct the EXACT V14 probability family
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))
def sigmoid(z):
    return 1/(1+np.exp(-z))

label_i={l:i for i,l in enumerate(LABELS)}
base_valid={}
base_test={}

base_valid["is_toxic"]=.875*v2_valid["is_toxic_prob"].to_numpy(float)+.125*old_xl_valid[1:3,:,0].mean(axis=0)
base_test["is_toxic"]=.875*v2_test["is_toxic_prob"].to_numpy(float)+.125*old_xl_test[1:3,:,0].mean(axis=0)

j=label_i["implicit_toxicity"]
base_valid["implicit_toxicity"]=sigmoid(.90*logit(v2_valid["implicit_toxicity_prob"].to_numpy(float))+.10*logit(old_bang_valid.mean(axis=0)[:,j]))
base_test["implicit_toxicity"]=sigmoid(.90*logit(v2_test["implicit_toxicity_prob"].to_numpy(float))+.10*logit(old_bang_test.mean(axis=0)[:,j]))

base_valid["sarcasm"]=v2_valid["sarcasm_prob"].to_numpy(float)
base_test["sarcasm"]=v2_test["sarcasm_prob"].to_numpy(float)

j=label_i["contains_profanity"]
base_valid["contains_profanity"]=.95*v2_valid["contains_profanity_prob"].to_numpy(float)+.05*old_bang_valid[0,:,j]
base_test["contains_profanity"]=.95*v2_test["contains_profanity_prob"].to_numpy(float)+.05*old_bang_test[0,:,j]

j=label_i["threatening"]
base_valid["threatening"]=.95*v2_valid["threatening_prob"].to_numpy(float)+.05*old_xl_valid[0,:,j]
base_test["threatening"]=.95*v2_test["threatening_prob"].to_numpy(float)+.05*old_xl_test[0,:,j]

j=label_i["identity_attack"]
base_valid["identity_attack"]=.70*v2_valid["identity_attack_prob"].to_numpy(float)+.30*old_bang_valid[2,:,j]
base_test["identity_attack"]=.70*v2_test["identity_attack_prob"].to_numpy(float)+.30*old_bang_test[2,:,j]

base_valid["sexual_content"]=v2_valid["sexual_content_prob"].to_numpy(float)
base_test["sexual_content"]=v2_test["sexual_content_prob"].to_numpy(float)

champ_thresholds={
    "is_toxic":0.6850110311061144,
    "implicit_toxicity":0.1416,
    "sarcasm":0.175,
    "contains_profanity":0.245,
    "threatening":0.16558668018085876,
    "identity_attack":0.3814,
    "sexual_content":0.245,
}

rebuilt=champion.copy()
for l in LABELS:
    rebuilt[l]=(base_test[l]>=champ_thresholds[l]).astype(np.int8)

diffs={l:int((rebuilt[l].to_numpy()!=champion[l].to_numpy()).sum()) for l in LABELS}
print("V14 reconstruction differences:",diffs)
assert sum(diffs.values())==0
rebuilt.to_csv(RUN_DIR/"v14_reconstructed_check.csv",index=False)
print("✅ Exact V14 reconstruction verified.")


In [ ]:

# 8. Prepare OOF predictions on the original 500-row validation
orig_valid_idx=np.arange(len(train),len(train)+len(valid))
muril_valid=oof[orig_valid_idx]

v17_bang_valid=v17_bang_oof[orig_valid_idx]
v17_xl_valid=v17_xl_oof[orig_valid_idx]
v17_mean_valid=.5*v17_bang_valid+.5*v17_xl_valid
v17_mean_test=.5*v17_bang_test+.5*v17_xl_test

fold_map=valid_folds.set_index("ID")["fold"]
selection_folds=valid["ID"].map(fold_map).to_numpy(int)
assert set(selection_folds)==set(range(N_FOLDS))
y_valid=valid[LABELS].to_numpy(np.int8)

print(pd.Series(selection_folds).value_counts().sort_index().to_dict())


In [ ]:

# 9. Competition-aligned per-label metrics + cross-fitted thresholds
def score_label(label,y,pred):
    if label=="is_toxic":
        return float(f1_score(y,pred,average="macro",labels=[0,1],zero_division=0))
    return float(f1_score(y,pred,zero_division=0))

def best_threshold(label,y,p):
    vals=np.unique(np.asarray(p,float))
    if len(vals)>600:
        vals=np.quantile(vals,np.linspace(0,1,600))
    cand=np.unique(np.r_[0.0,vals,1.000001])
    scores=np.array([score_label(label,y,p>=t) for t in cand])
    best=scores.max()
    tied=cand[np.isclose(scores,best,atol=1e-12,rtol=0)]
    return float(np.median(tied))

def crossfit(label,y,p):
    pred=np.zeros(len(y),dtype=np.int8)
    fs=[]; ts=[]
    for fold in range(N_FOLDS):
        held=selection_folds==fold
        t=best_threshold(label,y[~held],p[~held])
        ts.append(t)
        pred[held]=(p[held]>=t).astype(np.int8)
        fs.append(score_label(label,y[held],pred[held]))
    return {"score":score_label(label,y,pred),"fold_scores":np.asarray(fs,float),"thresholds":np.asarray(ts,float),"prediction":pred}

baseline_cv={}
for j,label in enumerate(LABELS):
    baseline_cv[label]=crossfit(label,y_valid[:,j],base_valid[label])

display(pd.DataFrame([
    {"label":l,"baseline_crossfit":baseline_cv[l]["score"],"fold_scores":np.round(baseline_cv[l]["fold_scores"],4).tolist()}
    for l in LABELS
]))


In [ ]:

# 10. Search conservative MuRIL blends
search_rows=[]
best_by_label={}

def blend(a,b,w,mode):
    if mode=="prob":
        return (1-w)*a+w*b
    return sigmoid((1-w)*logit(a)+w*logit(b))

for j,label in enumerate(LABELS):
    y=y_valid[:,j]
    base=base_valid[label]
    mu=muril_valid[:,j]
    bcv=baseline_cv[label]
    candidates=[]

    for mode in ["prob","logit"]:
        for w_i in range(1,13):  # 0.05...0.60
            w=w_i/20
            candidates.append({"kind":"global","mode":mode,"weight":w,"prob":blend(base,mu,w,mode)})

    if label=="is_toxic":
        for mode in ["prob","logit"]:
            for w_bn in [0.0,0.1,0.2,0.3,0.4]:
                for w_non in [0.1,0.2,0.3,0.4,0.5,0.6]:
                    if abs(w_non-w_bn)>0.4:
                        continue
                    weights=np.where(valid_has_bn,w_bn,w_non)
                    p=(1-weights)*base+weights*mu if mode=="prob" else sigmoid((1-weights)*logit(base)+weights*logit(mu))
                    candidates.append({"kind":"script_aware","mode":mode,"weight_bn":w_bn,"weight_non_bn":w_non,"prob":p})

        for mode in ["prob","logit"]:
            for wb,wm,wv in [
                (.8,.15,.05),(.75,.15,.10),(.7,.2,.1),
                (.7,.15,.15),(.65,.25,.10),(.6,.3,.1)
            ]:
                p=wb*base+wm*mu+wv*v17_mean_valid if mode=="prob" else sigmoid(wb*logit(base)+wm*logit(mu)+wv*logit(v17_mean_valid))
                candidates.append({"kind":"muril_plus_v17","mode":mode,"w_base":wb,"w_muril":wm,"w_v17":wv,"prob":p})

    best=None
    for c in candidates:
        p=c.pop("prob")
        cvres=crossfit(label,y,p)
        wins=int((cvres["fold_scores"]>bcv["fold_scores"]+1e-12).sum())
        losses=int((cvres["fold_scores"]<bcv["fold_scores"]-1e-12).sum())
        ties=N_FOLDS-wins-losses
        gain=cvres["score"]-bcv["score"]
        min_gain=TOXIC_MIN_GAIN if label=="is_toxic" else ATTRIBUTE_MIN_GAIN
        passes=(gain>=min_gain and wins>=MIN_WINS and losses<=MAX_LOSSES)

        rec={"label":label,**c,"score":cvres["score"],"baseline_score":bcv["score"],"gain":gain,"wins":wins,"ties":ties,"losses":losses,"passes_gate":passes,"median_threshold":float(np.median(cvres["thresholds"]))}
        search_rows.append(rec)

        if rec["kind"]=="global":
            intervention=rec["weight"]
        elif rec["kind"]=="script_aware":
            intervention=(rec["weight_bn"]+rec["weight_non_bn"])/2
        else:
            intervention=1-rec["w_base"]

        key=(passes,cvres["score"],wins,-losses,-intervention)
        if best is None or key>best["key"]:
            best={"key":key,"rec":rec,"cv":cvres}

    best_by_label[label]=best

search=pd.DataFrame(search_rows).sort_values(["passes_gate","gain","wins","losses"],ascending=[False,False,False,True])
search.to_csv(RUN_DIR/"v18_blend_search.csv",index=False)

best_summary=[]
for label,b in best_by_label.items():
    r=b["rec"].copy()
    r["weighted_score_gain"]=r["gain"]*(0.5 if label=="is_toxic" else 1/12)
    best_summary.append(r)

best_summary=pd.DataFrame(best_summary).sort_values("weighted_score_gain",ascending=False)
best_summary.to_csv(RUN_DIR/"v18_selection_summary.csv",index=False)
display(best_summary.round(5))


In [ ]:

# 11. Script diagnostics for is_toxic
j=LABELS.index("is_toxic")
base_pred=baseline_cv["is_toxic"]["prediction"]
mu_cv=crossfit("is_toxic",y_valid[:,j],muril_valid[:,j])

rows=[]
for name,mask in [("Bengali script",valid_has_bn),("No Bengali script",~valid_has_bn)]:
    if mask.sum():
        rows.append({
            "subset":name,
            "rows":int(mask.sum()),
            "positives":int(y_valid[mask,j].sum()),
            "V14_crossfit_macro_f1":score_label("is_toxic",y_valid[mask,j],base_pred[mask]),
            "MuRIL_crossfit_macro_f1":score_label("is_toxic",y_valid[mask,j],mu_cv["prediction"][mask]),
        })
script_report=pd.DataFrame(rows)
script_report.to_csv(RUN_DIR/"v18_script_report.csv",index=False)
display(script_report.round(4))


In [ ]:

# 12. Build controlled V18 submissions
def test_probability(label,rec):
    j=LABELS.index(label)
    base=base_test[label]
    mu=muril_test[:,j]

    if rec["kind"]=="global":
        return blend(base,mu,float(rec["weight"]),rec["mode"])

    if rec["kind"]=="script_aware":
        weights=np.where(test_has_bn,float(rec["weight_bn"]),float(rec["weight_non_bn"]))
        if rec["mode"]=="prob":
            return (1-weights)*base+weights*mu
        return sigmoid((1-weights)*logit(base)+weights*logit(mu))

    if rec["kind"]=="muril_plus_v17":
        wb=float(rec["w_base"]); wm=float(rec["w_muril"]); wv=float(rec["w_v17"])
        if rec["mode"]=="prob":
            return wb*base+wm*mu+wv*v17_mean_test
        return sigmoid(wb*logit(base)+wm*logit(mu)+wv*logit(v17_mean_test))

    raise ValueError(rec["kind"])

passed=best_summary[best_summary["passes_gate"]==True].copy()

def make_submission(labels_to_replace):
    out=champion.copy()
    details={}
    for label in labels_to_replace:
        rec=best_by_label[label]["rec"]
        p=test_probability(label,rec)
        threshold=float(np.median(best_by_label[label]["cv"]["thresholds"]))
        pred=(p>=threshold).astype(np.int8)
        before=out[label].to_numpy(int)
        out[label]=pred
        details[label]={
            "kind":rec["kind"],"mode":rec["mode"],"threshold":threshold,
            "changed_rows":int((before!=pred).sum()),
            "positives_before":int(before.sum()),"positives_after":int(pred.sum()),
            "crossfit_gain":float(rec["gain"]),
            "wins":int(rec["wins"]),"ties":int(rec["ties"]),"losses":int(rec["losses"]),
        }
    assert out["ID"].tolist()==champion["ID"].tolist()
    assert out[LABELS].isin([0,1]).all().all()
    return out,details

best_single_labels=[passed.iloc[0]["label"]] if len(passed) else []
all_passed_labels=passed["label"].tolist() if len(passed) else []
toxic_labels=["is_toxic"] if bool(best_by_label["is_toxic"]["rec"]["passes_gate"]) else []

single,single_details=make_submission(best_single_labels)
allp,all_details=make_submission(all_passed_labels)
toxic,toxic_details=make_submission(toxic_labels)

single.to_csv(RUN_DIR/"submission_v18_best_single.csv",index=False)
allp.to_csv(RUN_DIR/"submission_v18_all_passed.csv",index=False)
toxic.to_csv(RUN_DIR/"submission_v18_toxic_only.csv",index=False)

report={
    "version":"v18_muril_oof_multitask",
    "champion":"V14","champion_kaggle_score":0.586,
    "model":"google/muril-base-cased",
    "best_single_labels":best_single_labels,
    "all_passed_labels":all_passed_labels,
    "toxic_passed":bool(toxic_labels),
    "best_single_details":single_details,
    "all_passed_details":all_details,
    "toxic_details":toxic_details,
    "gates":{"toxic_min_gain":TOXIC_MIN_GAIN,"attribute_min_gain":ATTRIBUTE_MIN_GAIN,"min_wins":MIN_WINS,"max_losses":MAX_LOSSES},
    "warning":"OOF/development gains do not guarantee a Kaggle improvement."
}
(RUN_DIR/"v18_report.json").write_text(json.dumps(report,indent=2))

print("\nRECOMMENDATION")
if best_single_labels:
    print("Candidate passed. Best single label:",best_single_labels[0])
    print("Send v18_results.zip to ChatGPT BEFORE Kaggle submission.")
else:
    print("No MuRIL candidate passed. Keep V14 = 0.586.")
print("All passed labels:",all_passed_labels)
print("Toxic passed:",bool(toxic_labels))


In [ ]:

# 13. Package results
zip_path=Path("/content/v18_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)
from google.colab import files
files.download(str(zip_path))
